# Ноутбук 4 — Эксперимент 1: Baseline-модели arousal

**Назначение.** Воспроизведение CEAP-360VR baseline (LOSO accuracy ≈ 0,6426)
и демонстрация эффекта переноса на нового пользователя.

**Модели.** Logistic Regression, Random Forest (200 деревьев — exp01).

**Стратегии валидации.** Subject-Dependent 5-fold (верхняя оценка),
Leave-One-Subject-Out (32 фолда — реальное обобщение).

In [1]:
import sys
from pathlib import Path

NB_ROOT = Path.cwd()
if str(NB_ROOT) not in sys.path:
    sys.path.insert(0, str(NB_ROOT))

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time

from modules import config
from modules.experiments import (
    make_logistic_regression_window, make_random_forest_exp01,
    run_cv, NON_FEATURE_COLS,
)
from modules.validation import subject_dependent_splits, subject_independent_splits

TARGET = "arousal_class_bin"
print(f"Notebooks root: {NB_ROOT}")
print(f"Результаты: {config.RESULTS_DIR}")

Notebooks root: D:\Programming\Python\Diplom\notebooks
Результаты: D:\Programming\Python\Diplom\source\results


## 4.1 Загрузка матрицы признаков

In [2]:
df = pd.read_csv(config.RESULTS_DIR / "feature_table.csv")
feature_cols = [c for c in df.columns if c not in NON_FEATURE_COLS]
X = df[feature_cols].to_numpy(dtype=float)
y = df[TARGET].to_numpy(dtype=int)
print(f"Признаков: {len(feature_cols)}, окон: {len(df):,}")
print(f"Target '{TARGET}': {np.bincount(y).tolist()}")

Признаков: 143, окон: 15,104
Target 'arousal_class_bin': [8762, 6342]


## 4.2 Logistic Regression — SD 5-fold

In [3]:
t0 = time.time()
splits = list(subject_dependent_splits(df, y, n_folds=5))
rows_lr_sd = run_cv(make_logistic_regression_window, X, y, splits, "SD_5fold", "LogReg")
print(f"Время: {time.time()-t0:.1f} с")
acc = np.array([r['accuracy'] for r in rows_lr_sd])
f1 = np.array([r['f1_macro'] for r in rows_lr_sd])
print(f"  accuracy = {acc.mean():.4f} ± {acc.std():.4f}")
print(f"  f1_macro = {f1.mean():.4f} ± {f1.std():.4f}")

Время: 5.2 с
  accuracy = 0.6486 ± 0.0059
  f1_macro = 0.6194 ± 0.0090


## 4.3 Logistic Regression — LOSO (32 фолда)

In [4]:
t0 = time.time()
splits = list(subject_independent_splits(df))
rows_lr_loso = run_cv(make_logistic_regression_window, X, y, splits, "LOSO", "LogReg")
print(f"Время: {time.time()-t0:.1f} с")
acc = np.array([r['accuracy'] for r in rows_lr_loso])
f1 = np.array([r['f1_macro'] for r in rows_lr_loso])
auc = np.array([r['auc_roc'] for r in rows_lr_loso])
print(f"  accuracy = {acc.mean():.4f} ± {acc.std():.4f}")
print(f"  f1_macro = {f1.mean():.4f} ± {f1.std():.4f}")
print(f"  auc_roc  = {np.nanmean(auc):.4f} ± {np.nanstd(auc):.4f}")

Время: 26.0 с
  accuracy = 0.5961 ± 0.0802
  f1_macro = 0.5450 ± 0.0891
  auc_roc  = 0.6877 ± 0.0561


## 4.4 Random Forest — SD 5-fold

In [5]:
t0 = time.time()
splits = list(subject_dependent_splits(df, y, n_folds=5))
rows_rf_sd = run_cv(make_random_forest_exp01, X, y, splits, "SD_5fold", "RF")
print(f"Время: {time.time()-t0:.1f} с")
acc = np.array([r['accuracy'] for r in rows_rf_sd])
f1 = np.array([r['f1_macro'] for r in rows_rf_sd])
print(f"  accuracy = {acc.mean():.4f} ± {acc.std():.4f}")
print(f"  f1_macro = {f1.mean():.4f} ± {f1.std():.4f}")

Время: 4.8 с
  accuracy = 0.8475 ± 0.0062
  f1_macro = 0.8387 ± 0.0070


## 4.5 Random Forest — LOSO

In [6]:
t0 = time.time()
splits = list(subject_independent_splits(df))
rows_rf_loso = run_cv(make_random_forest_exp01, X, y, splits, "LOSO", "RF")
print(f"Время: {time.time()-t0:.1f} с")
acc = np.array([r['accuracy'] for r in rows_rf_loso])
f1 = np.array([r['f1_macro'] for r in rows_rf_loso])
auc = np.array([r['auc_roc'] for r in rows_rf_loso])
print(f"  accuracy = {acc.mean():.4f} ± {acc.std():.4f}")
print(f"  f1_macro = {f1.mean():.4f} ± {f1.std():.4f}")
print(f"  auc_roc  = {np.nanmean(auc):.4f} ± {np.nanstd(auc):.4f}")

Время: 37.0 с
  accuracy = 0.6322 ± 0.0651
  f1_macro = 0.5748 ± 0.0782
  auc_roc  = 0.6773 ± 0.0762


## 4.6 Сводная таблица результатов

In [7]:
all_rows = rows_lr_sd + rows_lr_loso + rows_rf_sd + rows_rf_loso
metrics_df = pd.DataFrame(all_rows)
summary = (
    metrics_df.groupby(["model", "strategy"])
    [["accuracy", "f1_macro", "auc_roc"]]
    .agg(["mean", "std"])
    .round(4)
)
print(summary)

# Сохраняем CSV для дальнейшего сравнения с exp02
metrics_df.to_csv(config.RESULTS_DIR / "exp01_metrics.csv", index=False)
summary.to_csv(config.RESULTS_DIR / "exp01_summary.csv")
print(f"\nСохранено: exp01_metrics.csv, exp01_summary.csv")

                accuracy         f1_macro         auc_roc        
                    mean     std     mean     std    mean     std
model  strategy                                                  
LogReg LOSO       0.5961  0.0815   0.5450  0.0905  0.6877  0.0570
       SD_5fold   0.6486  0.0066   0.6194  0.0100  0.6869  0.0072
RF     LOSO       0.6322  0.0661   0.5748  0.0795  0.6773  0.0775
       SD_5fold   0.8475  0.0069   0.8387  0.0078  0.9328  0.0049

Сохранено: exp01_metrics.csv, exp01_summary.csv


## 4.7 Выводы

* **Random Forest LOSO ≈ 0,63** — практически совпадает с опубликованным
  CEAP-360VR baseline (0,6426). Пайплайн валидирован.
* **Разрыв SD vs LOSO для RF составляет ≈21 п.п.** — деревья запоминают
  персон-специфичные паттерны при смешивании окон одного участника.
* **Logistic Regression** показывает значительно меньший разрыв (~5 п.п.) —
  линейная модель не имеет выразительности для запоминания.
* **Все модели упираются в потолок ≈0,65 LOSO accuracy** — таргет (глобальный
  порог arousal) связан с индивидуальным стилем самоотчёта.

Решение в ноутбуке 5: персональная Z-нормализация + персональная бинаризация.